# Training notebook for MobileNetV2

This notebook implements training for MobileNetV2 architectures for image classification.



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [3]:
import os
import random
import csv
import time

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from tqdm import tqdm
import time

from PIL import Image, ImageFilter
import cv2


## Reproducibility

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


Device: cuda


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [6]:
zip_path = "/content/drive/MyDrive/DLprojekt1/archive.zip"
!unzip -q "/content/drive/MyDrive/DLprojekt1/archive.zip" -d "/content/"

## Paths

In [7]:
# global variables and creating folders
DATA_DIR = "/content"

PROJECT_DIR = "/content/drive/MyDrive/DLprojekt1"

MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
PLOT_DIR = os.path.join(PROJECT_DIR, "plots")

PATIENCE = 5

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

RESULT_FILE = os.path.join(RESULT_DIR, "experiment_results_mobilenet.csv")

if not os.path.exists(RESULT_FILE):

    with open(RESULT_FILE,"w",newline="") as f:

        writer = csv.writer(f)
        writer.writerow([
            "experiment",
            "lr",
            "optimizer",
            "dropout",
            "weight_decay",
            "augmentation",
            "test_accuracy"
        ])


## Data Augmentations

In [8]:
class SobelAugmentation:

    def __init__(self, alpha=0.5):
        self.alpha = alpha

    def __call__(self, img):

        img_np = np.array(img)

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

        sobel = np.sqrt(sobelx**2 + sobely**2)

        sobel = np.uint8(255 * sobel / np.max(sobel))

        sobel_rgb = np.stack([sobel]*3, axis=-1)

        blended = cv2.addWeighted(img_np, 1-self.alpha, sobel_rgb, self.alpha, 0)

        return Image.fromarray(blended)


class GaussianBlur(object):

    def __call__(self,img):

        return img.filter(ImageFilter.GaussianBlur(radius=1.0))


def get_transform(augmentation=None, model_type="mobilenet"):

    transform_list = []

    if augmentation == "rotation":
        transform_list.append(transforms.RandomRotation(15))

    if augmentation == "color_jitter":
        transform_list.append(
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1
            )
        )

    if augmentation == "gaussian_blur":
        transform_list.append(GaussianBlur())

    if augmentation == "sobel":
        transform_list.append(SobelAugmentation())


    if model_type == "mobilenet":

      transform_list.extend([
          transforms.Resize((224, 224)),
          transforms.ToTensor(),
          transforms.Normalize(
              (0.485, 0.456, 0.406),
              (0.229, 0.224, 0.225)
          )
      ])

    else:

      transform_list.extend([
          transforms.ToTensor(),
          transforms.Normalize(
              (0.4789, 0.4723, 0.4305),
              (0.2421, 0.2383, 0.2587)
          )
      ])

    return transforms.Compose(transform_list)


def mixup_data(x, y, alpha=0.2):

    lam = np.random.beta(alpha,alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam*x + (1-lam)*x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, pred, y_a, y_b, lam):

    return lam*criterion(pred, y_a) + (1-lam)*criterion(pred, y_b)


## Data Loading

In [9]:
def load_data(augmentation=None, batch_size=256, subset_ratio=1.0, model_type="mobilenet"):

    train_transform = get_transform(augmentation, model_type)
    test_transform = get_transform(None, model_type)

    train_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "train"),
        transform=train_transform
    )

    valid_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "valid"),
        transform=test_transform
    )

    test_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "test"),
        transform=test_transform
    )

    if subset_ratio < 1.0:

        subset_size = int(len(train_set)*subset_ratio)
        train_set = torch.utils.data.Subset(train_set, range(subset_size))

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, valid_loader, test_loader


## MobileNetV2

In [10]:
class MobileNetV2Model(nn.Module):
    def __init__(self, dropout, num_classes=10, pretrained=True):
        super().__init__()

        self.model = models.mobilenet_v2(weights="IMAGENET1K_V1")

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)


    def set_trainable_layers(self, freeze_until):

        for i, layer in enumerate(self.model.features):
            requires_grad = i >= freeze_until
            for param in layer.parameters():
                param.requires_grad = requires_grad

## Evaluation

In [11]:

def evaluate(model, loader):

    model.eval()
    device = next(model.parameters()).device

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_correct / total, total_loss / total

In [12]:
class EarlyStopping:

    def __init__(self,patience=5):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def step(self,score):

        if self.best_score is None:
            self.best_score = score

        elif score <= self.best_score:

            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True

        else:

            self.best_score = score
            self.counter = 0

        return self.stop

## Training

In [13]:
def train_model(model, train_loader, val_loader, optimizer, epochs=30,
                use_mixup=False, name=None):

    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping(PATIENCE)

    history = {
        "train_loss":[],
        "val_loss":[],
        "val_acc":[],
    }

    best_acc = 0
    freeze_until = None

    for epoch in range(epochs):

        start_time = time.time()

        if epoch < 4:
            new_freeze = 16
        elif epoch < 8:
            new_freeze = 12
        else:
            new_freeze = 8

        if new_freeze != freeze_until:
            model.set_trainable_layers(new_freeze)

            optimizer = type(optimizer)(
                filter(lambda p: p.requires_grad, model.parameters()),
                **optimizer.defaults
            )

        freeze_until = new_freeze

        model.train()
        running_loss = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", unit="batch"):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            if use_mixup:

                images,targets_a,targets_b,lam = mixup_data(images,labels)
                outputs = model(images)
                loss = mixup_loss(criterion, outputs, targets_a, targets_b, lam)

            else:

                outputs = model(images)
                loss = criterion(outputs,labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            total += labels.size(0)

        val_acc, val_loss = evaluate(model, val_loader)
        train_loss = running_loss / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        epoch_time = time.time() - start_time

        print("train_loss:", f"{train_loss:.6f}", ", val_loss:", f"{val_loss:.6f}",
              ", val_acc:", f"{val_acc:.6f}", ", time:", round(epoch_time, 2), "s\n")

        torch.save(
            model.state_dict(),
            os.path.join(CHECKPOINT_DIR,f"checkpoint_epoch_{name}_{epoch+1}.pt")
        )

        if val_acc > best_acc:
            best_acc = val_acc

        if early_stopping.step(val_acc):

          print("Early stopping triggered")
          break

    return history


## Run Experiment

In [14]:
def save_history_csv(history, experiment_name):

    filename = os.path.join(RESULT_DIR, f"{experiment_name}_history.csv")

    epochs = len(history['train_loss'])

    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_loss', 'val_acc'])
        for i in range(epochs):
            writer.writerow([
                i+1,
                history['train_loss'][i],
                history['val_loss'][i],
                history['val_acc'][i]
            ])

    print(f"History saved in: {filename}")

In [15]:
def run_experiment(model_class, name, lr, optimizer_name, dropout, weight_decay, augmentation=None,
                   epochs=40, subset_ratio=1.0, use_mixup=False):

    print("Running:", name ,"\n")

    train_loader, val_loader, test_loader = load_data(
        augmentation=augmentation,
        subset_ratio=subset_ratio
    )

    model = model_class(dropout).to(device)

    if optimizer_name == "SGD":
        optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=lr,
                              weight_decay=weight_decay)

    elif optimizer_name == "SGD_momentum":
        optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, momentum=0.9, weight_decay=weight_decay)

    elif optimizer_name == "Adam":
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)

    history = train_model(model, train_loader, val_loader, optimizer, epochs=epochs, use_mixup=use_mixup, name=name)

    test_acc, _ = evaluate(model, test_loader)
    save_history_csv(history, name)

    torch.save(
        model.state_dict(),
        os.path.join(MODEL_DIR,f"{name}.pt")
    )

    with open(RESULT_FILE,"a",newline="") as f:

        writer = csv.writer(f)
        writer.writerow([
            name,
            lr,
            optimizer_name,
            dropout,
            weight_decay,
            augmentation,
            test_acc
        ])

    print("Test accuracy:", test_acc)


## Experiments

Experiments with different training, regularization parameters and data augumentation methods

For some experiments output might be not saved, but all of below experiments were run and their results saved and documented in the report.

In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_lr_0.1",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=8
)

Running: mobilenet_lr_0.1 



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 223MB/s]
Epoch 1:   0%|          | 0/352 [00:00<?, ?batch/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1: 100%|██████████| 352/352 [03:20<00:00,  1.76batch/s]


train_loss: 0.897787 , val_loss: 0.681058 , val_acc: 0.754278 , time: 380.03 s



...

train_loss: 0.330742 , val_loss: 0.597626 , val_acc: 0.798544 , time: 358.55 s



Epoch 7: 100%|██████████| 352/352 [03:07<00:00,  1.87batch/s]


train_loss: 0.246472 , val_loss: 0.559571 , val_acc: 0.821222 , time: 363.51 s



Epoch 8: 100%|██████████| 352/352 [03:10<00:00,  1.85batch/s]


train_loss: 0.177691 , val_loss: 0.650711 , val_acc: 0.808033 , time: 370.19 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_lr_0.1_history.csv
Test accuracy: 0.8071888888888888


In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_lr_0.01",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=8
)

Running: mobilenet_lr_0.01 



Epoch 1: 100%|██████████| 352/352 [03:01<00:00,  1.94batch/s]


train_loss: 1.116451 , val_loss: 0.833156 , val_acc: 0.706989 , time: 361.94 s



Epoch 2: 100%|██████████| 352/352 [03:04<00:00,  1.91batch/s]


train_loss: 0.806490 , val_loss: 0.747850 , val_acc: 0.734300 , time: 356.25 s



...

train_loss: 0.545455 , val_loss: 0.555621 , val_acc: 0.801111 , time: 359.2 s



Epoch 7: 100%|██████████| 352/352 [03:23<00:00,  1.73batch/s]


train_loss: 0.495092 , val_loss: 0.533721 , val_acc: 0.809256 , time: 386.48 s



Epoch 8: 100%|██████████| 352/352 [03:18<00:00,  1.77batch/s]


train_loss: 0.451650 , val_loss: 0.522727 , val_acc: 0.813478 , time: 372.65 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_lr_0.01_history.csv
Test accuracy: 0.8113888888888889


In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_lr_0.001",
    lr=0.001,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=8
)

Best lr parameter: 0.01

In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_optimizer_sgd_momentum",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=8
)

Running: mobilenet_optimizer_sgd_momentum 



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 195MB/s]
Epoch 1:   0%|          | 0/352 [00:00<?, ?batch/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1: 100%|██████████| 352/352 [03:15<00:00,  1.80batch/s]


train_loss: 0.824173 , val_loss: 0.688057 , val_acc: 0.753711 , time: 376.66 s



...

train_loss: 0.317742 , val_loss: 0.547843 , val_acc: 0.814767 , time: 374.03 s



Epoch 7: 100%|██████████| 352/352 [03:12<00:00,  1.83batch/s]


train_loss: 0.231171 , val_loss: 0.584885 , val_acc: 0.818756 , time: 374.02 s



Epoch 8: 100%|██████████| 352/352 [03:27<00:00,  1.70batch/s]


train_loss: 0.170397 , val_loss: 0.693201 , val_acc: 0.806144 , time: 386.74 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_optimizer_sgd_momentum_history.csv
Test accuracy: 0.8034333333333333


In [ ]:
#run_experiment(
#    model_class=MobileNetV2Model,
#    name="optimizer_sgd",
#    lr=0.01,
#    optimizer_name="SGD_momentum",
#    dropout=0.3,
#    weight_decay=1e-4,
#    epochs=20
#)

run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_optimizer_adam",
    lr=0.01,
    optimizer_name="Adam",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=8
)

Best optimizer parameter: SGD

In [ ]:
run_experiment(
   model_class=MobileNetV2Model,
   name="mobilenet_dropout_01",
   lr=0.01,
   optimizer_name="SGD",
   dropout=0.1,
   weight_decay=1e-4,
   epochs=8
)

Running: mobilenet_dropout_01 



Epoch 1: 100%|██████████| 352/352 [03:07<00:00,  1.87batch/s]


train_loss: 1.097715 , val_loss: 0.832351 , val_acc: 0.706156 , time: 361.85 s



Epoch 2: 100%|██████████| 352/352 [02:57<00:00,  1.99batch/s]


train_loss: 0.790175 , val_loss: 0.751668 , val_acc: 0.732478 , time: 356.05 s



...

train_loss: 0.533784 , val_loss: 0.559085 , val_acc: 0.799444 , time: 356.18 s



Epoch 7: 100%|██████████| 352/352 [03:02<00:00,  1.93batch/s]


train_loss: 0.481719 , val_loss: 0.533601 , val_acc: 0.809011 , time: 352.32 s



Epoch 8: 100%|██████████| 352/352 [03:08<00:00,  1.87batch/s]


train_loss: 0.438230 , val_loss: 0.522863 , val_acc: 0.813700 , time: 358.57 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_dropout_01_history.csv
Test accuracy: 0.8125


In [ ]:
# run_experiment(
#     model_class=MobileNetV2Model,
#     name="mobilenet_dropout_03",
#     lr=0.01,
#     optimizer_name="SGD",
#     dropout=0.3,
#     weight_decay=1e-4,
#     epochs=20
# )

run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_dropout_05",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.5,
    weight_decay=1e-4,
    epochs=8
)

Best dropout parameter: 0.3


In [16]:
run_experiment(
   model_class=MobileNetV2Model,
   name="mobilenet_weight_decay_1e2",
   lr=0.01,
   optimizer_name="SGD",
   dropout=0.3,
   weight_decay=1e-2,
   epochs=8
)

Running: mobilenet_weight_decay_1e2 



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 175MB/s]
Epoch 1:   0%|          | 0/352 [00:00<?, ?batch/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1: 100%|██████████| 352/352 [03:17<00:00,  1.78batch/s]


train_loss: 1.121577 , val_loss: 0.839344 , val_acc: 0.705344 , time: 369.41 s



...

train_loss: 0.551602 , val_loss: 0.559277 , val_acc: 0.801044 , time: 353.64 s



Epoch 7: 100%|██████████| 352/352 [03:13<00:00,  1.82batch/s]


train_loss: 0.500019 , val_loss: 0.532792 , val_acc: 0.809944 , time: 361.99 s



Epoch 8: 100%|██████████| 352/352 [03:03<00:00,  1.92batch/s]


train_loss: 0.456647 , val_loss: 0.516154 , val_acc: 0.815822 , time: 353.29 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_weight_decay_1e2_history.csv
Test accuracy: 0.8142333333333334


In [17]:
# run_experiment(
#     model_class=MobileNetV2Model,
#     name="mobilenet_weight_decay_1e3",
#     lr=0.01,
#     optimizer_name="SGD",
#     dropout=0.3,
#     weight_decay=1e-4,
#     epochs=10
# )

run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_dweight_decay_1e5",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-5,
    epochs=8
)

Running: mobilenet_dweight_decay_1e5 



Epoch 1: 100%|██████████| 352/352 [03:11<00:00,  1.84batch/s]


train_loss: 1.116453 , val_loss: 0.833163 , val_acc: 0.706978 , time: 362.85 s



Epoch 2: 100%|██████████| 352/352 [03:07<00:00,  1.88batch/s]


train_loss: 0.806488 , val_loss: 0.747841 , val_acc: 0.734300 , time: 361.15 s



...

train_loss: 0.545416 , val_loss: 0.555549 , val_acc: 0.801056 , time: 358.26 s



Epoch 7: 100%|██████████| 352/352 [03:05<00:00,  1.90batch/s]


train_loss: 0.495064 , val_loss: 0.533763 , val_acc: 0.809400 , time: 357.22 s



Epoch 8: 100%|██████████| 352/352 [03:05<00:00,  1.89batch/s]


train_loss: 0.451616 , val_loss: 0.522750 , val_acc: 0.813700 , time: 357.62 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_dweight_decay_1e5_history.csv
Test accuracy: 0.8112666666666667


Best weight_decay parameter: 1e-4

DATA AUGUMENTATION EXPERIMENTS

In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_augmentation_color_jitter",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="color_jitter",
    epochs=8
)

Running: mobilenet_augmentation_color_jitter 



Epoch 1: 100%|██████████| 352/352 [03:54<00:00,  1.50batch/s]


train_loss: 1.193514 , val_loss: 0.852502 , val_acc: 0.699144 , time: 429.21 s



Epoch 2: 100%|██████████| 352/352 [03:54<00:00,  1.50batch/s]


train_loss: 0.885251 , val_loss: 0.765259 , val_acc: 0.727811 , time: 414.52 s



...

train_loss: 0.626284 , val_loss: 0.569537 , val_acc: 0.795300 , time: 417.15 s



Epoch 7: 100%|██████████| 352/352 [04:08<00:00,  1.42batch/s]


train_loss: 0.580769 , val_loss: 0.542379 , val_acc: 0.805700 , time: 422.97 s



Epoch 8: 100%|██████████| 352/352 [04:01<00:00,  1.46batch/s]


train_loss: 0.544383 , val_loss: 0.526053 , val_acc: 0.811922 , time: 418.03 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_augmentation_color_jitter_history.csv
Test accuracy: 0.8109


In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_augmentation_blur",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="gaussian_blur",
    epochs=8
)

Running: mobilenet_augmentation_blur 



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 188MB/s]
Epoch 1:   0%|          | 0/352 [00:00<?, ?batch/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1: 100%|██████████| 352/352 [03:22<00:00,  1.74batch/s]


train_loss: 1.478700 , val_loss: 1.784787 , val_acc: 0.336867 , time: 405.05 s



...

train_loss: 0.884942 , val_loss: 1.739598 , val_acc: 0.425167 , time: 371.88 s



Epoch 7: 100%|██████████| 352/352 [03:11<00:00,  1.84batch/s]


train_loss: 0.821326 , val_loss: 1.727861 , val_acc: 0.428867 , time: 371.88 s



Epoch 8: 100%|██████████| 352/352 [03:14<00:00,  1.81batch/s]


train_loss: 0.767045 , val_loss: 1.784439 , val_acc: 0.419922 , time: 372.06 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_augmentation_blur_history.csv
Test accuracy: 0.41604444444444444


In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_augmentation_sobel",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="sobel",
    epochs=8
)

Running: mobilenet_augmentation_sobel 



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 80.3MB/s]
Epoch 1:   0%|          | 0/352 [00:00<?, ?batch/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1: 100%|██████████| 352/352 [03:53<00:00,  1.51batch/s]


train_loss: 1.318328 , val_loss: 1.020509 , val_acc: 0.637067 , time: 422.35 s



...

train_loss: 0.712544 , val_loss: 0.746768 , val_acc: 0.734700 , time: 437.35 s



Epoch 7: 100%|██████████| 352/352 [04:12<00:00,  1.39batch/s]


train_loss: 0.653890 , val_loss: 0.744988 , val_acc: 0.733100 , time: 441.6 s



Epoch 8: 100%|██████████| 352/352 [04:07<00:00,  1.42batch/s]


train_loss: 0.605863 , val_loss: 0.719352 , val_acc: 0.743978 , time: 439.76 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_augmentation_sobel_history.csv
Test accuracy: 0.7415888888888889


In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_augmentation_mixup",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    use_mixup=True,
    epochs=8
)

Running: mobilenet_augmentation_mixup 



Epoch 1: 100%|██████████| 352/352 [03:45<00:00,  1.56batch/s]


train_loss: 1.380508 , val_loss: 0.904071 , val_acc: 0.695533 , time: 421.97 s



Epoch 2: 100%|██████████| 352/352 [03:18<00:00,  1.77batch/s]


train_loss: 1.162690 , val_loss: 0.828669 , val_acc: 0.722822 , time: 386.28 s



...

train_loss: 0.989123 , val_loss: 0.644198 , val_acc: 0.791311 , time: 393.38 s



Epoch 7: 100%|██████████| 352/352 [03:45<00:00,  1.56batch/s]


train_loss: 0.964643 , val_loss: 0.615046 , val_acc: 0.794733 , time: 414.47 s



Epoch 8: 100%|██████████| 352/352 [03:40<00:00,  1.59batch/s]


train_loss: 0.901058 , val_loss: 0.596747 , val_acc: 0.805167 , time: 411.59 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_augmentation_mixup_history.csv
Test accuracy: 0.8047333333333333


In [ ]:
run_experiment(
    model_class=MobileNetV2Model,
    name="mobilenet_augmentation_rotation",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="rotation",
    epochs=8
)

Running: mobilenet_augmentation_rotation 



Epoch 1: 100%|██████████| 352/352 [03:41<00:00,  1.59batch/s]


train_loss: 1.278223 , val_loss: 0.876311 , val_acc: 0.688011 , time: 414.34 s



Epoch 2: 100%|██████████| 352/352 [03:41<00:00,  1.59batch/s]


train_loss: 0.954075 , val_loss: 0.782260 , val_acc: 0.720378 , time: 423.93 s



...

train_loss: 0.681810 , val_loss: 0.574481 , val_acc: 0.794089 , time: 418.65 s



Epoch 7: 100%|██████████| 352/352 [03:41<00:00,  1.59batch/s]


train_loss: 0.638141 , val_loss: 0.541357 , val_acc: 0.805844 , time: 420.31 s



Epoch 8: 100%|██████████| 352/352 [03:41<00:00,  1.59batch/s]


train_loss: 0.604323 , val_loss: 0.522656 , val_acc: 0.813056 , time: 422.11 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mobilenet_augmentation_rotation_history.csv
Test accuracy: 0.8103444444444444
